# Import packages

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import log_loss
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import LinearSVC
import plotly.express as px

# Load data

Import cleaned Congressional Bill data with dummy columns from data cleaning notebook.

In [ ]:
import pickle
from google.colab import drive

# connect to drive
drive.mount('/content/gdrive/')

# Load from file # Changed 'Save to file' to 'Load from file'
with open('xxx', 'rb') as file:
    bills_data = pickle.load(file)
    print('Done!')

In [ ]:
bills_data.head(3)

# Helper functions

## Confusion matrix visual

In [ ]:
from sklearn.metrics import confusion_matrix
import plotly.express as px

def plot_confusion_matrix(y_true, y_pred, height=450, showscale=False, title=None, subtitle=None):
    # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html
    # Confusion matrix whose i-th row and j-th column
    # ... indicates the number of samples with
    # ... true label being i-th class (ROW)
    # ... and predicted label being j-th class (COLUMN)
    cm = confusion_matrix(y_true, y_pred)

    class_names = sorted(y_test.unique().tolist())

    cm = confusion_matrix(y_test, y_pred, labels=class_names)

    title = title or "Confusion Matrix"
    if subtitle:
        title += f"<br><sup>{subtitle}</sup>"

    fig = px.imshow(cm, x=class_names, y=class_names, height=height,
                    labels={"x": "Predicted", "y": "Actual"},
                    color_continuous_scale="Blues", text_auto=True,
    )
    fig.update_layout(title={'text': title, 'x':0.485, 'xanchor': 'center'})
    fig.update_coloraxes(showscale=showscale)

    fig.show()

## Dataframe filter by type

In [ ]:
def filter_numeric_boolean(df):
    """
    Returns a DataFrame containing only columns of boolean, integer, or float types.

    Parameters:
      df (pandas.DataFrame): The input DataFrame.

    Returns:
      pandas.DataFrame: A new DataFrame with only boolean, integer, and float columns.
    """
    # 'number' covers all numeric types (integers and floats) and we add 'bool' explicitly.
    return df.select_dtypes(include=['bool', 'number'])

## AUC-ROC functions

In [ ]:
# AUC-ROC functions

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import roc_curve, roc_auc_score

## helpers

def _roc_stats(y_true, y_pred_proba):
    """
    Compute FPR, TPR, and AUC for one set of predictions.
    """
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    auc = roc_auc_score(y_true, y_pred_proba)
    return fpr, tpr, auc

def _add_random_ref(fig):
    """
    Add the diagonal 'random classifier' reference line to a Plotly figure.
    """
    fig.add_scatter(
        x=[0, 1], y=[0, 1],
        mode="lines",
        name="Random Classifier",
        line=dict(dash="dash"),
        showlegend=True
    )

def _add_auc_text(fig, auc_value, x=0.6, y=0.1):
    """
    Add a small textbox with the AUC score onto a Plotly figure.
    """
    fig.add_annotation(
        x=x, y=y,
        text=f"ROC‑AUC: {auc_value:.3f}",
        showarrow=False,
        font=dict(size=14, color="black"),
        bgcolor="white",
        bordercolor="black",
        borderwidth=1
    )


# Single curve function

def plot_roc_curve(
    y_true,
    y_pred_proba,
    title="ROC Curve",
    trace_name="ROC Curve"
):
    """
    Plot a single ROC curve

    Parameters
    ----------
    y_true : array‑like of shape (n_samples,)
    y_pred_proba : array‑like of shape (n_samples,)
        Predicted probabilities or scores for the positive class.
    title : str
    trace_name : str
    """
    fpr, tpr, auc_val = _roc_stats(y_true, y_pred_proba)

    roc_df = pd.DataFrame({"fpr": fpr, "tpr": tpr})
    fig = px.line(
        roc_df,
        x="fpr", y="tpr",
        title=title,
        labels={"fpr": "False Positive Rate", "tpr": "True Positive Rate"},
    )

    fig.update_traces(name=trace_name, showlegend=True, selector=lambda _: True)
    _add_random_ref(fig)
    _add_auc_text(fig, auc_val)

    return fig

# Multi-curve function (multi-trace)

def plot_multi_roc(
    y_true,
    model_preds,
    title="Combined ROC Curves"
):
    """
    Plot ROC curves for multiple models on the same axes.

    Parameters
    ----------
    y_true : array‑like of shape (n_samples,)
    model_preds : Iterable[Tuple[str, array‑like]]
        Each tuple contains (model_label, y_pred_proba_array).
    title : str
    """
    fig = go.Figure()

    for label, probs in model_preds:
        fpr, tpr, auc_val = _roc_stats(y_true, probs)
        fig.add_scatter(
            x=fpr,
            y=tpr,
            mode="lines",
            name=f"{label} (AUC={auc_val:.3f})"
        )

    _add_random_ref(fig)

    fig.update_layout(
        title=title,
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate",
        legend_title_text="Models"
    )

    return fig

## Precision-recall functions

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import precision_recall_curve, average_precision_score

# helpers

def _pr_stats(y_true, y_pred_proba):
    """
    Compute precision, recall, and Average Precision (AP) for one set of predictions.
    """
    precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)
    ap = average_precision_score(y_true, y_pred_proba)
    return precision, recall, ap


def _add_baseline_ref(fig, pos_rate, name="Positive Prevalence"):
    """
    Add a horizontal reference line at the dataset's positive class prevalence.
    """
    fig.add_scatter(
        x=[0, 1],
        y=[pos_rate, pos_rate],
        mode="lines",
        name=name,
        line=dict(dash="dash"),
        showlegend=True
    )


def _add_ap_text(fig, ap_value, x=0.6, y=0.1):
    """
    Annotate a Plotly figure with the Average Precision score.
    """
    fig.add_annotation(
        x=x, y=y,
        text=f"Avg. Precision: {ap_value:.3f}",
        showarrow=False,
        font=dict(size=14, color="black"),
        bgcolor="white",
        bordercolor="black",
        borderwidth=1
    )

# Single curve

def plot_pr_curve(
    y_true,
    y_pred_proba,
    title="Precision‑Recall Curve",
    trace_name="Precision‑Recall Curve"
):
    """
    Plot a single Precision–Recall curve (keeps backward compatibility).

    Parameters
    ----------
    y_true : array‑like
    y_pred_proba : array‑like
    title : str
    trace_name : str
    """
    precision, recall, ap_val = _pr_stats(y_true, y_pred_proba)

    pr_df = pd.DataFrame({"Recall": recall, "Precision": precision})
    fig = px.line(
        pr_df,
        x="Recall",
        y="Precision",
        title=title,
        labels={"Recall": "Recall", "Precision": "Precision"}
    )

    fig.update_traces(name=trace_name, showlegend=True, selector=lambda _: True)
    _add_baseline_ref(fig, pos_rate=y_true.mean())
    _add_ap_text(fig, ap_val)

    return fig

# Multi-curve (multi-trace)

def plot_multi_pr(
    y_true,
    model_preds,
    title="Combined Precision‑Recall Curves"
):
    """
    Plot Precision–Recall curves for multiple models on the same axes.

    Parameters
    ----------
    y_true : array‑like
    model_preds : Iterable[Tuple[str, array‑like]]
        Each tuple contains (model_label, y_pred_proba_array).
    title : str
    """
    fig = go.Figure()

    for label, probs in model_preds:
        precision, recall, ap_val = _pr_stats(y_true, probs)
        fig.add_scatter(
            x=recall,
            y=precision,
            mode="lines",
            name=f"{label} (AP={ap_val:.3f})"
        )

    _add_baseline_ref(fig, pos_rate=y_true.mean())

    fig.update_layout(
        title=title,
        xaxis_title="Recall",
        yaxis_title="Precision",
        legend_title_text="Models"
    )

    return fig

# Data preprocessing

## Data exploration

In [ ]:
print(f'Number of bills: {bills_data.shape[0]:,}')
print(f'Number of features / attributes: {bills_data.shape[1]:,}')

In [ ]:
reported_bill_count = bills_data[bills_data['reported_to_floor'] == 1].shape[0]

print(f'The dataset contains {reported_bill_count:,} bills reported to the floor.')

In [ ]:
%%capture output
bills_data.columns.tolist()

In [ ]:
bills_data.columns.tolist()

In [ ]:
# Drop columns with list values
bills_data_clean = bills_data.drop(['cosponsor_parties', 'cosponsor_states',
                                    'committees_house', 'sub-committees_house',
                                    'subject_areas'], axis = 1)

## Target relocation

In [ ]:
# move target column to front of dataframe
cols = list(bills_data_clean.columns)
cols.insert(0, cols.pop(cols.index('reported_to_floor')))
bills_data_clean = bills_data_clean.loc[:, cols]
bills_data_clean.head(3)

## Drop columns not fit for modeling (e.g, strings)

In [ ]:
drop_columns = [
    'title',
    'bill_number',
    'body',
    'bill_type',
    'status',
    'sponsor_party',
    'sponsor_people_id',
    'sponsor_hash',
    'sponsor_first_name',
    'sponsor_last_name',
    'sponsor_state',
    'name_state',
    'recent_chamber',
    'name',
    'start_year',
    'last_name',
    'first_name',
    'state',
    'abbreviation',
    'introduction_date',
    'bill_type_mapped',
    'bill_number_clean',
    'bill_id_concat',
]

# Drop the columns from the DataFrame
bills_data_clean = bills_data_clean.drop(columns=drop_columns)

## Unique values

In [ ]:
# Print number of unique values in each series
bills_data_clean.nunique()

## NA value handling

In [ ]:
# Count number of records with NA values
bills_data_clean.isna().sum().sort_values(ascending=False)

In [ ]:
# Remove all records with NA values

bills_data_clean = bills_data_clean.dropna()

In [ ]:
# Count number of records with NA values
bills_data_clean.isna().sum().sort_values(ascending=False)

## Processed data summary

In [ ]:
print(f"Number of bills: {bills_data_clean.shape[0]:,}")
print(f"Number of features / attributes: {bills_data_clean.shape[1]:,}")
print(f"Number of bills reported to floor remaining in dataset: {bills_data_clean['reported_to_floor'].sum():,}")

In [ ]:
bills_data_clean.columns.to_list()

# Modelling

### Model parameters

In [ ]:
# number of folds for cross-validation
num_folds = 10
test_size = 0.2
score_method = 'average_precision' # or neg-log-loss or auc-roc

## Model data preparation

### Define x/y

In [ ]:
def filter_numeric_boolean(df):
    """
    Returns a DataFrame containing only columns of boolean, integer, or float types.

    Parameters:
      df (pandas.DataFrame): The input DataFrame.

    Returns:
      pandas.DataFrame: A new DataFrame with only boolean, integer, and float columns.
    """
    # 'number' covers all numeric types (integers and floats) and we add 'bool' explicitly.
    return df.select_dtypes(include=['bool', 'number'])

In [ ]:
# Filter dataframe for numeric/boolean/dummy coded columns

bills_data_clean_model = filter_numeric_boolean(bills_data_clean)

In [ ]:
# Select target (y) for the model
y = bills_data_clean_model['reported_to_floor']

In [ ]:
# Maximum number of x attributes (ints/floats/booleans)
x = bills_data_clean_model.drop(['reported_to_floor'], axis=1)

### Scale x

In [ ]:
# Scale X data

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# store column names
original_columns = bills_data_clean_model.drop(['reported_to_floor'], axis=1).columns

scaler = StandardScaler()
x = scaler.fit_transform(x)

### Train/test split

In [ ]:
# Create train/test split

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=99, test_size=test_size)
print(f'TRAIN/TEST SPLIT SIZE:')
print("TRAIN:", x_train.shape, y_train.shape)
print("TEST:", x_test.shape, y_test.shape)

## Logistic regression

### Base logistic regression (no feature selection)


#### Train model

In [ ]:
# Create logistic regression

from sklearn.linear_model import LogisticRegression
import pandas as pd

model = LogisticRegression(class_weight='balanced')

model.fit(x_train, y_train)

#### Predict on test set

In [ ]:
# Predict on test set
y_pred = model.predict(x_test)

#### Calculate classification matrix

In [ ]:
# Print metrics: accuracy, recall, precision and f1
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, output_dict=False))

base_log_classification = classification_report(y_test, y_pred, output_dict=True)

In [ ]:
from sklearn.metrics import confusion_matrix
import plotly.express as px

def plot_confusion_matrix(y_true, y_pred, height=450, showscale=False, title=None, subtitle=None):
    # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html
    # Confusion matrix whose i-th row and j-th column
    # ... indicates the number of samples with
    # ... true label being i-th class (ROW)
    # ... and predicted label being j-th class (COLUMN)
    cm = confusion_matrix(y_true, y_pred)

    class_names = sorted(y_test.unique().tolist())

    cm = confusion_matrix(y_test, y_pred, labels=class_names)

    title = title or "Confusion Matrix"
    if subtitle:
        title += f"<br><sup>{subtitle}</sup>"

    fig = px.imshow(cm, x=class_names, y=class_names, height=height,
                    labels={"x": "Predicted", "y": "Actual"},
                    color_continuous_scale="Blues", text_auto=True,
    )
    fig.update_layout(title={'text': title, 'x':0.485, 'xanchor': 'center'})
    fig.update_coloraxes(showscale=showscale)

    fig.show()

In [ ]:
base_log_classification_plot = plot_confusion_matrix(y_test, y_pred, height=450, showscale=False, title="Baseline Logistic Regression Confusion Matrix", subtitle=None)

base_log_classification_plot

#### Calculate ROC-AUC

In [ ]:
# Get ROC-AUC for model
from sklearn.metrics import roc_auc_score

# get "logits" (predicted probabilities for each class)
y_pred_proba = model.predict_proba(x_test)[:,1]

# save to dataframe
y_pred_proba_df = pd.DataFrame(y_pred_proba, columns=['base_logistic'])

# for multi-class, pass all probas and use "ovr" (one vs rest)
base_log_roc_auc = roc_auc_score(y_test, y_pred_proba)
print("ROC-AUC, base logistic regression:", base_log_roc_auc)

In [ ]:
y_pred_proba_df.head()

In [ ]:
base_log_roc_plot = plot_roc_curve(y_test, y_pred_proba, title = 'ROC Curve: Base Logistic', trace_name = "Base Logistic")

base_log_roc_plot

#### Calculate precision-recall

In [ ]:
# Compute the Average Precision score for the model
from sklearn.metrics import average_precision_score # Import the missing function

base_log_avg_precision = average_precision_score(y_test, y_pred_proba)
print("Average Precision, base logistic regression:", base_log_avg_precision)

In [ ]:
base_log_pr_plot = plot_pr_curve(y_test, y_pred_proba, title = "Precision-recall curve: Base Logisitic", trace_name = "Base Logistic")

base_log_pr_plot

#### Calculate loss

In [ ]:
cost_base_log = log_loss(y_test, y_pred_proba)

print(f'Log-Loss (Cost): {cost_base_log}')

### Lasso regression

#### Train model

In [ ]:
# Initialize logistic regression with L1 regularization
logreg = LogisticRegression(penalty='l1', solver='saga', max_iter=2500, class_weight = 'balanced')

# Define the parameter grid for the regularization strength C
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10]}


# Set up GridSearchCV with x-fold cross validation and ROC-AUC as scoring
grid_search_lasso = GridSearchCV(logreg, param_grid, cv=num_folds, scoring=score_method, n_jobs = -1, verbose = 3)
grid_search_lasso.fit(x_train, y_train)

In [ ]:
# Get the optimal model from grid search
optimal_model_lasso = grid_search_lasso.best_estimator_

In [ ]:
print("Best parameters:", grid_search_lasso.best_params_)
print(f'Best cross-validation {score_method} score: {grid_search_lasso.best_score_:.3f}')

# Identify features with non-zero coefficients
nonzero_idx = np.where(optimal_model_lasso.coef_.ravel() != 0)[0]
selected_features = original_columns[nonzero_idx]

print("Selected features using optimal L1 Regularization:")
print(selected_features)

#### Predict on test set

In [ ]:
# Predict on test set
y_pred = optimal_model_lasso.predict(x_test)

#### Calculate classification matrix

In [ ]:
# Print metrics: accuracy, recall, precision and f1
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, output_dict=False))

lasso_log_classification = classification_report(y_test, y_pred, output_dict=True)

In [ ]:
lasso_log_classification_plot = plot_confusion_matrix(y_test, y_pred, height=450, showscale=False, title="Lasso Logistic Regression Confusion Matrix", subtitle=None)

lasso_log_classification_plot

#### Calculate ROC-AUC

In [ ]:
# Get ROC-AUC for model
from sklearn.metrics import roc_auc_score

# get "logits" (predicted probabilities for each class)
y_pred_proba = optimal_model_lasso.predict_proba(x_test)[:,1]

# save to dataframe
y_pred_proba_df['lasso'] = y_pred_proba

# for multi-class, pass all probas and use "ovr" (one vs rest)
lasso_roc_auc = roc_auc_score(y_test, y_pred_proba)
print("ROC-AUC:", lasso_roc_auc)

In [ ]:
lasso_roc_plot = plot_roc_curve(y_test, y_pred_proba, title = 'ROC Curve: Lasso regression', trace_name="Lasso")

lasso_roc_plot

#### Calculate precision-recall

In [ ]:
# Compute the Average Precision score for the model
lasso_avg_precision = average_precision_score(y_test, y_pred_proba)
print("Average Precision, lasso regression:", lasso_avg_precision)

In [ ]:
lasso_reg_pr_plot = plot_pr_curve(y_test, y_pred_proba, title = "Precision-recall curve: Lasso regression",  trace_name="Lasso")

lasso_reg_pr_plot

#### Calculate loss

In [ ]:
cost_base_lasso = log_loss(y_test, y_pred_proba)

print(f'Log-Loss (Cost): {cost_base_lasso}')

### Ridge regression

This should look like a lasso regression 1-1 just using the l2 penalty type and tuning alpha across the grid

This is the default penalty in the sklearn logistic regression package (so we can tune alpha and use it to better select our variables)

#### Train Model

In [ ]:
# Initialize logistic regression with L2 regularization
logreg = LogisticRegression(penalty='l2', solver='saga', max_iter=2500, class_weight = 'balanced')

# Define the parameter grid for the regularization strength C
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10]}


# Set up GridSearchCV with x-fold cross validation and ROC-AUC as scoring
grid_search_ridge = GridSearchCV(logreg, param_grid, cv=num_folds, scoring=score_method, n_jobs = -1, verbose = 3)
grid_search_ridge.fit(x_train, y_train)

#### Get optimal model

In [ ]:
# Get the optimal model from grid search
optimal_model_ridge = grid_search_ridge.best_estimator_

In [ ]:
print("Best parameters:", grid_search_ridge.best_params_)
print(f'Best cross-validation {score_method} score: {grid_search_ridge.best_score_:.3f}')

# Identify features with non-zero coefficients
nonzero_idx = np.where(optimal_model_ridge.coef_.ravel() != 0)[0]
selected_features = original_columns[nonzero_idx]

print("Selected features using optimal L2 Regularization:")
print(selected_features)

#### Predict on test set

In [ ]:
# Predict on test set
y_pred = optimal_model_ridge.predict(x_test)

#### Calculate classification matrix

In [ ]:
# Print metrics: accuracy, recall, precision and f1
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, output_dict=False))

ridge_log_classification = classification_report(y_test, y_pred, output_dict=True)

In [ ]:
ridge_log_classification_plot = plot_confusion_matrix(y_test, y_pred, height=450, showscale=False, title="Ridge Logistic Regression Confusion Matrix", subtitle=None)

ridge_log_classification_plot

#### Calculate ROC-AUC

In [ ]:
# Get ROC-AUC for model
from sklearn.metrics import roc_auc_score

# get "logits" (predicted probabilities for each class)
y_pred_proba = optimal_model_ridge.predict_proba(x_test)[:,1]

# save to dataframe
y_pred_proba_df['ridge'] = y_pred_proba

# for multi-class, pass all probas and use "ovr" (one vs rest)
ridge_roc_auc = roc_auc_score(y_test, y_pred_proba)
print("ROC-AUC:", ridge_roc_auc)

In [ ]:
ridge_roc_plot = plot_roc_curve(y_test, y_pred_proba, title = 'ROC Curve: Ridge regression', trace_name="Ridge")

ridge_roc_plot

#### Calculate precision-recall

In [ ]:
# Compute the Average Precision score for the model
ridge_avg_precision = average_precision_score(y_test, y_pred_proba)
print("Average Precision, ridge regression:", ridge_avg_precision)

In [ ]:
ridge_reg_pr_plot = plot_pr_curve(y_test, y_pred_proba, title = "Precision-recall curve: Ridge regression",  trace_name="Ridge")

ridge_reg_pr_plot

#### Calculate loss

In [ ]:
cost_base_ridge = log_loss(y_test, y_pred_proba)

print(f'Log-Loss (Cost): {cost_base_ridge}')

## SVM (sklearn)

#### Train Model

In [ ]:
# Create a Pipeline that first selects the best k features, then fits the SVC.
pipeline = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_classif)),  # Feature selection step
    ('svc', SVC(probability=True, class_weight='balanced', max_iter=10000, tol=1e-4))  # SVC classifier step
])

param_grid = {
    # Tune the number of features to select (you can also use 'all' to keep all features)
    'feature_selection__k': [20, 40, 60, 120],

    # SVC parameters:
    'svc__C': [0.001, 0.1, 1],
    'svc__kernel': ['linear', 'rbf']
    # # gamma is only relevant for non-linear kernels (e.g., 'rbf')
    # 'svc__gamma': ['scale', 'auto', 0.001, 0.01, 0.1]
}

# Set up GridSearchCV with cross validation and the defined scoring method.
grid_search_svc = GridSearchCV(
    pipeline,
    param_grid,
    cv=num_folds,           # num_folds should be defined (e.g., 5 or 10)
    scoring=score_method,   # score_method should be defined (e.g., 'roc_auc')
    n_jobs=-1,
    verbose=3
)

# Fit the grid search on your training data.
grid_search_svc.fit(x_train, y_train)

# After grid search completes, you can examine the best parameters.
print("Best parameters found:", grid_search_svc.best_params_)
print("Best cross-validation score:", grid_search_svc.best_score_)

#### Get optimal model

In [ ]:
# Get the optimal model from grid search
optimal_model_SVC1 = grid_search_svc.best_estimator_

In [ ]:
print("Best parameters:", grid_search_svc.best_params_)
print(f'Best cross-validation {score_method} score: {grid_search_svc.best_score_:.3f}')


In [ ]:
# Get the best pipeline from grid search
selector = grid_search_svc.best_estimator_.named_steps['feature_selection']

# Get the boolean mask of selected features
mask = selector.get_support()

# Use the original feature names, not the ndarray
selected_columns = original_columns[mask]
print("Selected features:", selected_columns)

#### Predict on test set

In [ ]:
# Predict on test set
y_pred = optimal_model_SVC1.predict(x_test)

#### Calculate classification matrix

In [ ]:
# Print metrics: accuracy, recall, precision and f1
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, output_dict=False))

SVM1_classification = classification_report(y_test, y_pred, output_dict=True)

In [ ]:
plot_confusion_matrix(y_test, y_pred, height=450, showscale=False, title= "SVM Confusion Matrix", subtitle=None)

#### Calculate ROC-AUC

In [ ]:
# Get ROC-AUC for model

# get "logits" (predicted probabilities for each class)
y_pred_proba = optimal_model_SVC1.predict_proba(x_test)[:,1]

# save to dataframe
y_pred_proba_df['SVM'] = y_pred_proba

# for multi-class, pass all probas and use "ovr" (one vs rest)
SVM1_roc_auc = roc_auc_score(y_test, y_pred_proba)
print("ROC-AUC:", SVM1_roc_auc)

In [ ]:
SVM1_roc_plot = plot_roc_curve(y_test, y_pred_proba, title = 'ROC Curve: SVM1 regression', trace_name= "SVM")

SVM1_roc_plot

#### Calculate precision-recall

In [ ]:
# Compute the Average Precision score for the model
SVM1_avg_precision = average_precision_score(y_test, y_pred_proba)
print("Average Precision, SVM1 regression:", SVM1_avg_precision)

In [ ]:
SVM1_pr_plot = plot_pr_curve(y_test, y_pred_proba, title = "Precision-recall curve: SVM1 regression",  trace_name="Lasso")

SVM1_pr_plot

#### Calculate loss

In [ ]:
cost_base_SVM1 = log_loss(y_test, y_pred_proba)

print(f'Log-Loss (Cost): {cost_base_SVM1}')

## Neural network (feed forward)

In [ ]:
!pip install keras-tuner

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Sequential, Input
import keras_tuner as kt
from keras_tuner import Objective
# Assume X_train and y_train are already defined and preprocessed.
input_dim = x_train.shape[1]

def build_model(hp):
    model = Sequential()

    # 0) Explicit Input layer to avoid the input_dim warning
    model.add(Input(shape=(input_dim,)))

    # 1) First hidden layer: tune number of units and dropout rate
    units_1 = hp.Choice('units_1', values=[32, 64, 128], default=64)
    model.add(layers.Dense(units=units_1, activation='relu'))

    # dropout layer
    dropout_1 = hp.Float('dropout_1', min_value=0.0, max_value=0.7, step=0.1, default=0.5)
    model.add(layers.Dropout(rate=dropout_1))

    # 2) Second hidden layer: tune number of units and dropout rate
    units_2 = hp.Choice('units_2', values=[16, 32, 64], default=32)
    model.add(layers.Dense(units=units_2, activation='relu'))
    dropout_2 = hp.Float('dropout_2', min_value=0.0, max_value=0.7, step=0.1, default=0.5)
    model.add(layers.Dropout(rate=dropout_2))

    # 3) Output layer
    model.add(layers.Dense(1, activation='sigmoid'))

    # 4) Tune the learning rate for the optimizer
    learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4], default=1e-3)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[
            # ROC‑AUC
            tf.keras.metrics.AUC(name='roc_auc'),
            # PR‑AUC
            tf.keras.metrics.AUC(name='pr_auc', curve='PR'),
            'accuracy'
        ]
    )
    return model


# Set up the tuner
tuner = kt.RandomSearch(
    build_model,
    objective=Objective("val_pr_auc", direction="max"),  # Optimize for PR‑AUC
    max_trials=20,             # Number of hyperparameter configurations to try
    executions_per_trial=1,    # How many times to train each config
    directory='my_tuner_dir',  # Where to save tuning logs
    project_name='binary_classification'
)

# Summarize the search space
tuner.search_space_summary()

# Run the hyperparameter search
tuner.search(x_train, y_train, epochs=50, validation_split=0.2, verbose=3)

# Summarize results
tuner.results_summary()

# Retrieve the best model and hyperparameters
best_model = tuner.get_best_models(num_models=1)[0]
best_hps   = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best hyperparameters:", best_hps.values)

In [ ]:
# Predict on test set
y_pred = best_model.predict(x_test)

# convert to binary with threshold
threshold = 0.5
y_pred = (y_pred_proba >= threshold).astype(int).ravel()

#### Calculate classification matrix

In [ ]:
# Print metrics: accuracy, recall, precision and f1
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, output_dict=False))

NN_classification = classification_report(y_test, y_pred, output_dict=True)



In [ ]:
base_log_classification_plot = plot_confusion_matrix(y_test, y_pred, height=450, showscale=False, title="Neural Network Confusion Matrix", subtitle=None)


#### Calculate ROC-AUC

In [ ]:
# Get ROC-AUC for model

# get predicted probabilities
y_pred_proba = best_model.predict(x_test)

# save to dataframe
y_pred_proba_df['NN'] = y_pred_proba

# for multi-class, pass all probas and use "ovr" (one vs rest)
NN_roc_auc = roc_auc_score(y_test, y_pred_proba)
print("ROC-AUC:", NN_roc_auc)

In [ ]:
NN_roc_plot = plot_roc_curve(y_test, y_pred_proba, title = 'ROC Curve: Neural Net', trace_name= "Neural Net")

NN_roc_plot

#### Calculate precision-recall

In [ ]:
# Compute the Average Precision score for the model
NN_avg_precision = average_precision_score(y_test, y_pred_proba)
print("Average Precision, Neural Net:", NN_avg_precision)

In [ ]:
NN_pr_plot = plot_pr_curve(y_test, y_pred_proba, title = "Precision-recall curve: Neural Net",  trace_name="Neural Net")

NN_pr_plot

#### Calculate loss

In [ ]:
cost_NN = log_loss(y_test, y_pred_proba)

print(f'Log-Loss (Cost): {cost_NN}')

#### Get optimal model

# Analysis

## AUC-ROC plot

In [ ]:
# create multi-trace AUC-ROC plot

multi_roc_auc = plot_multi_roc(y_test, [
    ("Base logistic", y_pred_proba_df['base_logistic']),
    ("Lasso regression", y_pred_proba_df['lasso']),
    ("Ridge regression", y_pred_proba_df['ridge']),
    ("SVM", y_pred_proba_df['SVM']),
    ("Neural Net", y_pred_proba_df['NN'])
], title = "ROC Curves")

multi_roc_auc

## Precision-Recall plot

In [ ]:
# create multi-trace AUC-ROC plot

multi_pr_plot = plot_multi_pr(y_test, [
    ("Base logistic", y_pred_proba_df['base_logistic']),
    ("Lasso regression", y_pred_proba_df['lasso']),
    ("Ridge regression", y_pred_proba_df['ridge']),
    ("SVM", y_pred_proba_df['SVM']),
    ("Neural Net", y_pred_proba_df['NN'])
], title = "Precision-Recall Curves")

multi_pr_plot

In [ ]:
# classification summary

# List of dictionaries for different models
model_metrics = [base_log_classification,
                 lasso_log_classification,
                 ridge_log_classification,
                 SVM1_classification,
                 NN_classification]


# Flatten the nested dictionaries into a DataFrame.
df_metrics = pd.json_normalize(model_metrics)


# Define an identifier for each model (e.g., model names or numbers)
identifiers = ["Base Logistic Regression", "Lasso Regression", "Ridge Regression","SVM Classification", "Neural Net"]

# Add the identifier column to the DataFrame
df_metrics['model'] = identifiers

df_metrics

# move identifier to front
cols = df_metrics.columns.tolist()
cols = cols[-1:] + cols[:-1]
df_metrics = df_metrics[cols]

df_metrics

In [ ]:
# Reshape for Plotly
df_melted = df_metrics.melt(id_vars="model", value_vars=["True.precision", "True.recall", "True.f1-score", "accuracy"],
                    var_name="Metric", value_name="Score")
rename_map = {
    "True.precision": "Precision (True)",
    "True.recall": "Recall (True)",
    "True.f1-score": "F1-score (True)",
    "accuracy": "Accuracy"
}

df_melted["Metric"] = df_melted["Metric"].replace(rename_map)

# Create bar chart
fig = px.bar(df_melted, x="Metric", y="Score", color="model", barmode="group",
             color_discrete_sequence=px.colors.qualitative.Set3,
             title="Confusion Metrics",
             labels={"Score": "Metric Value", "model": "Model", "Metric": "Confusion Metric"})

# Update trace to show text only on first group, make it larger, rotated, and above bars
fig.update_traces(
    textposition="outside",
    textangle=180,  # rotate upside down
    textfont_size=14  # larger font
)

fig.show()